## Question brief
- **Stakeholder:** A small Indie Studio Looking to Release their new game on Steam
- **Decision:** What price point would be best to use for their game
- **Question:** Within the Indie RPG genre, which price range gets the best player reception?
- **Metrics:** Review score (positive ÷ total ratings), for games with 50+ ratings
- **Not answering:** Whether players will buy and play their new game

## Data trust
- **Source:** Snapshot of Steam Store in 2019 by Nik Davis
- **License:** CC BY 4.0, Free to publish, with credit to the author

| Caveat | What it means for this analysis |
|---|---|
| Snapshot from 2019 | This should be used as a guide only, player expectations and prices may have changed |
| Owner counts are estimated ranges | Instead of Owners Use Review because its a direct account from the player |
| Prices are in GBP | Charts are labelled in £; converting to other currencies is approximate |
| Only games with 50+ ratings are included | Any game with less Reviews won't be seen. With regards to the targeted genre, Indie RPG, about 50% of games did not reach the ratings threshold and are excluded. So the results describe Indie RPGs that have already found an audience, not **ALL** Indie RPGs. |

In [1]:
import pandas as pd
df = pd.read_csv("data/steam.csv")
df.shape


(27075, 18)

In [2]:
# Reception metric: share of ratings that are positive
df["total_ratings"] = df["positive_ratings"] + df["negative_ratings"]
df["review_score"] = df["positive_ratings"] / df["total_ratings"]

# WHERE total_ratings >= 50: a score from 3 ratings is noise
rated = df[df["total_ratings"] >= 50]
len(rated)

11818

In [3]:
# CASE WHEN price = 0 THEN 'Free' WHEN price <= 5 THEN ... END
rated = rated.assign(price_band=pd.cut(
    rated["price"],
    bins=[-1, 0, 5, 10, 20, 500],
    labels=["Free", "Under £5", "£5–10", "£10–20", "£20+"]))

In [6]:
# Prediction: More expensive games aren't always received better than cheaper games, but they are more likely to be received well. Let's see if this is true.
# SELECT price_band, COUNT(*), PERCENTILE_CONT(0.5) ... GROUP BY price_band
summary = rated.groupby("price_band", observed=True).agg(
    games=("appid", "count"),
    median_score=("review_score", "median"),
    median_price=("price", "median"),
    median_ratings=("total_ratings", "median"),
).round(2)
summary

,games,median_score,median_price,median_ratings
price_band,,,,
Free,1561,0.74,0.00,295.0
Under £5,4148,0.75,2.79,144.5
£5–10,2765,0.80,6.99,235.0
£10–20,2583,0.82,14.99,380.0
£20+,761,0.78,29.99,974.0


## First finding
- The scores rise from **0.74 (free)** and peak at **0.82 at £10–20** then it drops again to **0.78 at £20+**.
- My prediction was actually right. The most expensive games don't always get the best ratings. Next, I'll check this with the genre filter, look at small groups, and test whether price actually affects reviews.
- This is across all game genres; Indie RPG alone may look different.
- The £20+ group reaches far more players, which could be due to most being AAA games from bigger studios. Some reviews could be influenced by this as well as purchases.

## Indie RPG deep-dive

In [7]:
# WHERE genres Like '%Indie%' AND genres LIKE '%RPG%'
is_indie_rpg = (df["genres"].str.contains("Indie", regex=False)
                & df["genres"].str.contains("RPG", regex=False))
indie_rpg_all = df[is_indie_rpg]
len(indie_rpg_all)

3215

In [ ]:
# Expectation: I thought only ~25% of Indie RPGs would reach 50+ ratings.
# Result: about 50% do (50.3% fall below the threshold).

# AVG(CASE WHEN total_rating < 50 THEN 1 ELSE 0 END)
f"{(indie_rpg_all['total_ratings'] < 50).mean():.0%}"


'50%'

In [11]:
indie_rpg = indie_rpg_all[indie_rpg_all["total_ratings"] >= 50]
indie_rpg = indie_rpg.assign(price_band=pd.cut(
    indie_rpg["price"],
    bins=[-1, 0, 5, 10, 20, 500],
    labels=["Free", "Under £5", "£5–10", "£10–20", "£20+"]))

indie_rpg.groupby("price_band", observed=True).agg(
    games=("appid", "count"),
    median_score=("review_score", "median"),
    median_ratings=("total_ratings", "median"),
).round(2)

,games,median_score,median_ratings
price_band,,,
Free,182,0.72,283.0
Under £5,504,0.76,144.5
£5–10,413,0.79,244.0
£10–20,457,0.81,351.0
£20+,43,0.76,840.0


In [12]:
import numpy as np
rng = np.random.default_rng(42)  # fixed seed: same result every run

def bootstrap_ci(scores, n=2000):
    medians = [np.median(rng.choice(scores, size=len(scores), replace=True))
               for _ in range(n)]
    return np.percentile(medians, [2.5, 97.5])

In [14]:
# Prediciton: the widest range will be the £20+ because there are fewer games in that price range, 
# so the median is less stable. The narrowest range will be the £5–10 because there are more games in that price range, 
# so the median is more stable.

rows = []
for band, group in indie_rpg.groupby("price_band", observed=True):
    low, high = bootstrap_ci(group["review_score"].to_numpy())
    width = high - low
    rows.append({"price_band": band, "games": len(group),
                 "median_score": group["review_score"].median(),
                 "ci_low": low, "ci_high": high, "ci_width": width})

ci_table = pd.DataFrame(rows).round(2)
ci_table

,price_band,games,median_score,ci_low,ci_high,ci_width
0,Free,182,0.72,0.70,0.75,0.05
1,Under £5,504,0.76,0.73,0.79,0.05
2,£5–10,413,0.79,0.78,0.81,0.03
3,£10–20,457,0.81,0.80,0.82,0.02
4,£20+,43,0.76,0.69,0.81,0.12


## Is the pattern real?

- The price pattern held for Indie RPG: scores rise from **0.72** (free) to **0.81 at £10–20**. The £20+ median is 0.76, but its range (0.69–0.81) is so wide that I wouldn't trust it, because the data doesn't support a drop.
- The difference in the Free and £10-20 range is real and the £5-10 vs £10-20 overlap so we can't separate them.
- This data excludes Indie RPGs with fewer than 50 ratings, about half the genre. This only shows how successful Indie RPGs were rated, not whether their price caused that success. 
- Based on 2019 Steam data, I'd suggest pricing in the 10-20 range, because it had the highest score and the narrowest range. I'm confident that Indie RPGs priced £10–20 were rated higher than free ones, but the data can't tell us whether £5–10 was better than £10–20, or how £20+ compares.